In [1]:
import os
#script_dir = os.path.dirname(__file__)
#os.chdir(script_dir)

import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import minmax_scale

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import joblib
import pandas as pd

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Load dataset
def load_dataset(dataset):
    data = pd.read_csv(dataset)
    X = data.iloc[:, :12].values
    y = data["target"].values

    # Oversample using SMOTE
    X, y = SMOTE(random_state=42).fit_resample(X, y)
    return data, X, y


def test_train_split(X, y):
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    # Scaling
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    # Dump scaler # We need this to use pre-trained model
    # pre-trained model require same scaler as training scaler
    joblib.dump(sc, 'scaler.pkl')

    return X_train, y_train


In [3]:
data, X, y = load_dataset('../heart_dataset.csv')
feature_names = list(data.columns[:12])

X_train, y_train = test_train_split(X, y)

In [4]:
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

In [5]:
print(list(data.columns[:12]))

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca']


In [6]:
explainer = shap.Explainer(rf)
shap_values = explainer(X_train, check_additivity=False)
shap_values_class1 = shap_values.values[:, :, 1]

In [7]:
import numpy as np

rf_importance = rf.feature_importances_
mi_importance = mutual_info_classif(X_train, y_train)
shap_importance = np.abs(shap_values_class1).mean(axis=0)

In [8]:
combined_score = (
    minmax_scale(rf_importance) +
    minmax_scale(mi_importance) +
    minmax_scale(shap_importance)
) / 3
ranked_features = np.argsort(combined_score)[::-1]
feature_names = list(data.columns[:12])

print(ranked_features)

[10  3  4 11  2  9  6  7  0  1  5  8]


In [9]:
hyperparameter_ranges = {
    "n1": (128, 192),
    "n2": (64, 128),
    "n3": (32, 64),
    "learning_rate": (0.0001, 0.01),
    "dropout_rate": (0.0, 0.4),
    "l2_regularization": (0.000001, 0.01),
    "alpha": (0.01, 0.3)
}

import random

def hyprparameters(hyprparameter_ranges):

    hyperparameter = []

    for key, value in hyprparameter_ranges.items():
        low = value[0]
        high = value[1]

        if isinstance(low, int):
            random_value = random.randint(low, high)
        elif isinstance(low, float):
            random_value = max(low, random.uniform(max(0, low), max(0, high)))

        hyperparameter.append(random_value)

    return hyperparameter



#counts = 0
# for key, value in hyperparameter_ranges.items():
#    if key.startswith('n') and key[1:].isdigit():
#        counts += 1

#for count in range(counts):
#    count+=1
#    print(f"model: {count} layer")
    

In [10]:
def smart_initial_feature_mask(k=8):
    mask = [0] * 12
    for idx in ranked_features[:k]:
        mask[idx] = 1
    return mask


def smart_individual():
    feature_mask = smart_initial_feature_mask(k=random.randint(6, 10))
    hyperparams = hyprparameters(hyperparameter_ranges)
    return feature_mask + hyperparams

In [11]:
print(smart_individual())

[1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 141, 127, 60, 0.003641875449462293, 0.06688658146728757, 0.006041427446596412, 0.15993279578617728]


In [12]:
def smart_individual():
    k = random.randint(6, 10)
    print(f"Random K: {k}")
    
    feature_mask = [0] * 12
    
    for idx in ranked_features[:k]:
        feature_mask[idx] = 1
    
    hyperparams = hyprparameters(hyperparameter_ranges)
    return feature_mask + hyperparams

print(smart_individual())

Random K: 6
[0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 166, 78, 33, 0.003216752618753094, 0.3307623203006966, 0.0036263886420676472, 0.14312500109091048]


In [13]:
hyperparameter_ranges = {
    "n1": (128, 192),
    "n2": (64, 128),
    "n3": (32, 64),
    "learning_rate": (0.0001, 0.01),
    "dropout_rate": (0.0, 0.4),
    "l2_regularization": (0.000001, 0.01),
    "alpha": (0.01, 0.3)
}

import random

def hyprparameters(hyprparameter_ranges):

    hyperparameter = []

    for key, (low, high) in hyprparameter_ranges.items():

        if isinstance(low, int) and isinstance(high, int):
            random_value = random.randint(low, high)
        elif isinstance(low, float) and isinstance(high, float):
            random_value = max(low, random.uniform(max(0, low), max(0, high)))

        hyperparameter.append(random_value)

    return hyperparameter

parameters = hyprparameters(hyperparameter_ranges)
print(parameters)

[135, 107, 64, 0.0014900672180512826, 0.3878791680483299, 0.007814161242073679, 0.14473960643606054]


In [14]:
import numpy as np

def smart_individual():
    k = random.randint(6, 10)
    print(f"Random K: {k}")
    
    feature_mask = [0] * 12
    
    for idx in ranked_features[:k]:
        feature_mask[idx] = 1
    
    hyperparams = hyprparameters(hyperparameter_ranges)
    return feature_mask + hyperparams

def repair(individual):
    hyprparameter = individual[12:]
    feature_mask = individual[:12]

    # Feature Selection
    for i in range(12):
        if feature_mask[i] in (0, 1):
            feature_mask[i] = int(individual[i])
        else:
            feature_mask[i] = 1

    if sum(feature_mask[:12]) == 0:
        feature_mask[random.randint(0, 11)] = 1

    # Hyprparameters
    for i, (key, (low, high)) in enumerate(hyperparameter_ranges.items()):
        hyprparameter[i] = np.clip(hyprparameter[i], low, high)

    return feature_mask + hyprparameter

#individual = smart_individual()

individual = [0, 0, 2, 1, 3, 0, 1, 1, 0, 1, 1, 1, 2000, 82, 57, 0.0007012195905534492, 0.14939781406884373, 0.004934872527467051, 0.035310347529296834]
repaired = repair(individual)

print(f"Individual: {individual}")
print(f"Repaired:   {repaired}")

Individual: [0, 0, 2, 1, 3, 0, 1, 1, 0, 1, 1, 1, 2000, 82, 57, 0.0007012195905534492, 0.14939781406884373, 0.004934872527467051, 0.035310347529296834]
Repaired:   [0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, np.int64(192), np.int64(82), np.int64(57), np.float64(0.0007012195905534492), np.float64(0.14939781406884373), np.float64(0.004934872527467051), np.float64(0.035310347529296834)]


In [15]:
data.shape[1] - (1 if 'target' in data.columns else 0)

12

In [16]:
data.drop(['target'], axis=1).shape[1]

12

In [17]:
parameters = {
    "dataset": '../heart_dataset.csv',
    "target": ( 'target' )
}
features = data.drop(parameters["target"], axis=1).shape[1]
data[[parameters["target"]]].shape[1]

fearure_names = data.columns[data.columns != parameters["target"]].tolist()

data[feature_names].values

input_shape = len(feature_names)
print(input_shape)

12


In [20]:
hyperparameter_ranges = {
    "l1": (128, 192),
    "l2": (64, 128),
    "l3": (32, 64),
    "learning_rate": (0.0001, 0.01),
    "dropout_rate": (0.0, 0.4),
    "l2_regularization": (0.000001, 0.01),
    "alpha": (0.01, 0.3)
}

layer_key = []
hidden_layer = 0
for key in hyperparameter_ranges:
     if key.startswith('l') and key[1:].isdigit():
         layer_key.append(key)
         hidden_layer += 1

print(f"layer_keys: {layer_key}")
layer_units = params[:hidden_layer]
for idx, val in enumerate(layer_value, start=1):
    locals()[f"l{idx}"] = val

lr, dr, l2_reg, alpha = params[hidden_layer:]
print(f"{l1, l2, l3, lr, dr, l2_reg, alpha}")

layer_keys: ['l1', 'l2', 'l3']


NameError: name 'params' is not defined

In [19]:
for i, key in enumerate(layer_keys):


_IncompleteInputError: incomplete input (167480333.py, line 1)

In [21]:
hyperparameter_ranges = {
    "l1": (128, 192),
    "l2": (64, 128),
    "l3": (32, 64),
    "learning_rate": (0.0001, 0.01),
    "dropout_rate": (0.0, 0.4),
    "l2_regularization": (0.000001, 0.01),
    "alpha": (0.01, 0.3)
}

len(hyperparameter_ranges)

7

In [26]:
for i in range(1,2 + 1):
    print(i)


1
2


In [6]:
import joblib
import pandas as pd

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import minmax_scale

def load_dataset(dataset, target_list):
    data = pd.read_csv(dataset)
    feature_names = data.drop(columns=target_list).columns.tolist()
    target_names = target_list

    return {
        "data": data,
        "features": feature_names,
        "target": target_names
    }

def load_Xy(data, feature, target):
    X = data[feature].values
    y = data[target].values

    return { "X":X, "y":y }

def test_train_split(X, y):    
    # Oversample using SMOTE
    X, y = SMOTE(random_state=42).fit_resample(X, y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    # Scaling
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return {
        "X_train":X_train,
        "y_train":y_train,
        "X_test":X_test,
        "y_test":y_test
    }


def data_dictionary(dataset_path, target_list):
    data = load_dataset(dataset_path, target_list)
    data.update(load_Xy(data["data"], data["feature"], data["target"]))
    data.update(test_train_split(data["X"], data["y"]))

    return data


dataset_path = '../heart_dataset.csv'
target_list = ['target']

dataset = data_dictionary(dataset_path, target_list)
print("Keys:", dataset.keys())
#print(data)
#print("Values:", data.values())

# Dump scaler # We need this to use pre-trained model
    # pre-trained model require same scaler as training scaler
    #joblib.dump(sc, 'models/scaler.pkl')


Keys: dict_keys(['data', 'feature', 'target', 'X', 'y', 'X_train', 'y_train', 'X_test', 'y_test'])


In [9]:
print(dataset["target"])

['target']
